# Indic TTS phrase cache

One canonical cache key for Indic synthesis requests, so that two spellings of the same
phrase do not cost two calls.

**Nothing in this notebook has been run against the live API.** There was no Sarvam API key
on the machine it was written on. Every cell up to the synthesis cell is offline and covered
by tests; the one cell that calls the API ships with an empty output and has never been
executed. Run it yourself before trusting what it would print.

Pipeline overview:

1. Build a canonical key from the text plus every parameter that changes the audio.
2. Replay an invented 46-request log through the key, one normalisation layer at a time,
   and count the calls each layer saves.
3. Size a disk cache from the eviction table.
4. Synthesise one phrase for real, store it, and watch a differently-spelled request for the
   same phrase hit the entry with no second call.

The demo log is **invented**. No native speaker has reviewed its Hindi or Odia.

In [ ]:
%pip install -r requirements.txt

## Setup

This cell needs no key and no network. The whole measurement below runs offline.

In [ ]:
from __future__ import annotations

import base64
import os
import sys
from pathlib import Path

RECIPE_DIR = Path.cwd()
if str(RECIPE_DIR) not in sys.path:
    sys.path.insert(0, str(RECIPE_DIR))

from tts_cache import (
    DEFAULT_LAYERS,
    LAYER_ORDER,
    OFF_BY_DEFAULT,
    NormalisationPolicy,
    PhraseCache,
    SynthesisRequest,
    canonical_key,
    canonical_text,
    format_ladder,
    layer_ladder,
    replay,
)
from demo_log import DEMO_LOG, ODIA_PHONE_DECOMPOSED, ODIA_PHONE_PRECOMPOSED

OUTPUT_DIR = RECIPE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("requests in the demo log:", len(DEMO_LOG))
print("distinct texts          :", len({one.text for one in DEMO_LOG}))

## The problem, in four lines

Two spellings of the Devanagari letter FA. They look identical, they are meant to be read
identically, and they are different byte strings. Both are conforming Unicode; which one you
get depends on the keyboard, the CMS and the paste buffer.

Every varying character below is built from an explicit code point. A pasted glyph is exactly
what this recipe exists to disambiguate, and an editor can normalise one of a pair without
telling you.

In [ ]:
precomposed = "अपना " + chr(0x095E) + chr(0x094B) + chr(0x0928) + " नंबर"
decomposed = "अपना " + chr(0x092B) + chr(0x093C) + chr(0x094B) + chr(0x0928) + " नंबर"

print("look the same :", precomposed, "|", decomposed)
print("equal strings :", precomposed == decomposed)

byte_exact = NormalisationPolicy.none()
one = SynthesisRequest(text=precomposed, language_code="hi-IN")
two = SynthesisRequest(text=decomposed, language_code="hi-IN")

print("two keys, byte-exact :", canonical_key(one, byte_exact) != canonical_key(two, byte_exact))
print("one key, default     :", canonical_key(one) == canonical_key(two))

## The layers, and why three of them are off

A layer is on by default only when the difference it folds cannot change what is spoken by
any conforming engine: either Unicode itself calls the two forms the same text, or the
characters removed are invisible format controls with no phonetic role.

A false miss costs one API call. A false hit plays the wrong audio to somebody who cannot see
the screen. Those are not symmetric, so nothing is folded on a hunch.

- `nukta_fold` is off because a nukta is phonemic. Devanagari JA and ZA are different
  consonants, and so are Odia DDA and RRA.
- `zero_width_joiner` is off because U+200C and U+200D change which conjunct is rendered.
  This recipe's position is that they should not change the sound, but a position is not a
  measurement, and settling it needs an API call we could not make.
- `digit_form` is off because whether native digits and ASCII digits reach the same phonemes
  is unverifiable from here. The SDK's own documentation warns that `10000` and `10,000` are
  read differently, which is direct evidence that surface form matters to the engine.

`punctuation_tail` is on, and it is the one default resting on an **assumption** rather than
a definition: that a trailing danda and a trailing full stop are the same end-of-statement
mark. The double danda is deliberately never folded, and text ending in no terminator keeps
its own key.

In [ ]:
print("order   :", LAYER_ORDER)
print("on      :", sorted(DEFAULT_LAYERS))
print("off     :", sorted(OFF_BY_DEFAULT))

danda = chr(0x0964)
samples = {
    "doubled spaces": "  " + chr(0x0915) + "  " + chr(0x0916) + "  ",
    "zero-width space": chr(0x0915) + chr(0x200B) + " " + " " + chr(0x0916),
    "full stop for a danda": "धन्यवाद.",
    "double danda, never folded": "धन्यवाद" + chr(0x0965),
}
for label, text in samples.items():
    print("%-28s %r -> %r" % (label, text, canonical_text(text)))

## What each layer is worth

The ladder switches the layers on one at a time, in order, and replays the whole log after
each one. `additional calls saved` is that layer's own contribution.

Rung 0 is what a byte-exact cache would have achieved. The gap between rung 0 and the default
policy is the part of the saving that comes from the key rather than from the cache.

In [ ]:
ladder = layer_ladder(DEMO_LOG, max_entries=1000)
print(format_ladder(ladder))

default_result = replay(DEMO_LOG, NormalisationPolicy.default(), max_entries=1000)
byte_exact_result = replay(DEMO_LOG, NormalisationPolicy.none(), max_entries=1000)

print()
print("byte-exact caching : %d calls" % byte_exact_result.misses)
print("default policy     : %d calls" % default_result.misses)
print("saved by the key   : %d calls" % (byte_exact_result.misses - default_result.misses))

### The merge this recipe believes is wrong

Rung 2 folds the nuktas, and one of the three merges it makes is between two different Odia
words: the one written with RRA (U+0B5C) and the one written with plain DDA (U+0B21). Those
are different consonants.

That pair is in the demo log on purpose. The ladder is only honest if it shows you what you
would be buying when you switch a layer on.

In [ ]:
vehicle_rra = chr(0x0B17) + chr(0x0B3E) + chr(0x0B5C) + chr(0x0B3F)
vehicle_dda = chr(0x0B17) + chr(0x0B3E) + chr(0x0B21) + chr(0x0B3F)

folding = NormalisationPolicy.default().with_layer("nukta_fold")
odia_rra = SynthesisRequest(text=vehicle_rra, language_code="od-IN")
odia_dda = SynthesisRequest(text=vehicle_dda, language_code="od-IN")

print("different words   :", vehicle_rra, "|", vehicle_dda)
print("default: two keys :", canonical_key(odia_rra) != canonical_key(odia_dda))
print("folded : one key  :", canonical_key(odia_rra, folding) == canonical_key(odia_dda, folding))

## How big does the cache need to be

Small. The log contains 16 distinct keys under the default policy, and a 13-entry cache loses
nothing on this traffic: the three entries it evicts are never asked for again.

In [ ]:
print("%-12s %6s %8s %10s %10s" % ("max_entries", "hits", "misses", "evictions", "resident"))
for size in (4, 8, 10, 13, 16, 64):
    sized = replay(DEMO_LOG, NormalisationPolicy.default(), max_entries=size)
    print("%-12d %6d %8d %10d %10d"
          % (size, sized.hits, sized.misses, sized.evictions, sized.final_size))

## The disk cache

Content-addressed: one file per key, named for the key, plus a small JSON index. Recency is
an integer tick rather than a wall clock, because two writes in the same millisecond would
tie on a timestamp and make eviction depend on dictionary order.

A truncated, extended, missing or corrupt entry is always a miss and never a crash: the entry
is dropped from the index and `stats.dropped` counts it.

`outputs/` is gitignored, so nothing you synthesise here can be committed by accident.

In [ ]:
cache = PhraseCache(OUTPUT_DIR / "phrase_cache", max_entries=13)

print("index :", cache.index_path)
print("audio :", cache.audio_dir)
print("entries already stored:", len(cache))

## The one cell that needs a key

Everything above ran offline. From here on you need a Sarvam API key in `.env`.

The key is passed **explicitly**. `SarvamAI.__init__` reads `SARVAM_API_KEY` in a default
argument, which Python evaluates once at import time, so a `load_dotenv()` call that runs
after the import is too late and the client raises.

In [ ]:
from dotenv import load_dotenv
from sarvamai import SarvamAI

load_dotenv()

if not os.getenv("SARVAM_API_KEY"):
    raise RuntimeError(
        "Set SARVAM_API_KEY in your environment or .env file before running this cell."
    )

client = SarvamAI(api_subscription_key=os.environ["SARVAM_API_KEY"])
print("client ready")

### Synthesise one phrase and store it

This is the only cell in the notebook that calls the API. It has never been run, so its
output is empty.

Note what goes over the wire: the **original** text, untouched. The canonical form was only
ever used to find the entry. That is what makes folding survivable at all.

`enable_cached_responses` is not sent. Its docstring says the server-side cache is available
on `bulbul:v1` and `bulbul:v2` only, `bulbul:v1` is not in the SDK's model Literal, and
`bulbul:v2` is deprecated in `scripts/sarvam_api_rules.json`. On `bulbul:v3` there is nothing
for it to act on.

In [ ]:
odia_request = SynthesisRequest(
    text=ODIA_PHONE_PRECOMPOSED,
    language_code="od-IN",
    model="bulbul:v3",
    speaker="anushka",
)

print("sending:", odia_request.to_convert_kwargs())

response = client.text_to_speech.convert(**odia_request.to_convert_kwargs())
audio_bytes = base64.b64decode(response.audios[0])

stored_key = cache.put(odia_request, audio_bytes)
(OUTPUT_DIR / "odia_phone_prompt.wav").write_bytes(audio_bytes)

print("bytes stored:", len(audio_bytes))
print("under key   :", stored_key)

### The payoff

The same Odia prompt, written with the vowel sign O decomposed into E plus AA instead of the
single U+0B4B. A different byte string, the same word, the same sound. The cache serves it
from the entry that was just stored, with no second call.

In [ ]:
same_phrase_other_spelling = SynthesisRequest(
    text=ODIA_PHONE_DECOMPOSED,
    language_code="od-IN",
    model="bulbul:v3",
    speaker="anushka",
)

print("different bytes:", ODIA_PHONE_PRECOMPOSED != ODIA_PHONE_DECOMPOSED)
print("same key       :", canonical_key(same_phrase_other_spelling) == stored_key)

served = cache.get(same_phrase_other_spelling)
print("served from disk, no API call:", served == audio_bytes)
print("stats:", cache.stats)

cache.flush()